# Use Model Context Protocol (MCP) as tools with Strands Agents

## Overview
The [Model Context Protocol (MCP)](https://modelcontextprotocol.io/introduction) is an open protocol that standardizes how applications provide context to Large Language Models (LLMs). Strands Agents SDK integrates with MCP to extend agent capabilities through external tools and services.

MCP enables communication between agents and MCP servers that provide additional tools. The Strands Agents SDK includes built-in support for connecting to MCP servers and using their tools.

In this example we will show you how to use MCP tools on your Strands Agent. We will use the [AWS Documentation MCP server](https://awslabs.github.io/mcp/servers/aws-documentation-mcp-server) which provides tools to access AWS documentation, search for content, and get recommendations. This MCP server has 3 main features:

- **Read Documentation**: Fetch and convert AWS documentation pages to markdown format
- **Search Documentation**: Search AWS documentation using the official search API
- **Recommendations**: Get content recommendations for AWS documentation pages



## Tutorial Details

| Information      | Details                                         |
|:-----------------|:------------------------------------------------|
| Agent structure  | Single agent                                    |
| Model provider   | Amazon Bedrock                                  |
| Model            | Anthropic Claude Sonnet 4.5                     |
| Tools            | MCP tools                                       |
| Strands features | MCPClient, stdio and Streamable HTTP transports |

## Architecture

<div style="text-align:center">
    <img src="images/architecture.png" width="65%" />
</div>

## What you'll learn
* Connect an agent to an MCP server over stdio
* Connect over Streamable HTTP
* Use tools from multiple MCP servers with one agent
* Invoke MCP tools directly

## Setup and prerequisites

### Prerequisites
* Python 3.10+
* AWS account
* Anthropic Claude Sonnet 4.5 enabled on Amazon Bedrock

Let's now install the requirement packages for our agent

In [ ]:
# installing pre-requisites
!pip install -r requirements.txt

### Importing dependency packages

Now let's import the dependency packages

In [1]:
import threading
import time
from datetime import timedelta

from mcp import StdioServerParameters, stdio_client
from mcp.client.streamable_http import streamable_http_client
from mcp.server import MCPServer
from strands import Agent
from strands.tools.mcp import MCPClient

### Connect to MCP server using stdio transport

[Transports](https://modelcontextprotocol.io/specification/2025-03-26/basic/transports) in MCP provide the foundations for communication between clients and servers. It handles the underlying mechanics of how messages are sent and received. At the moment there are three standards transport implementations built-in in MCP:

- **Standard Input/Output (stdio)**: enables communication through standard input and output streams. It is particularly useful for local integrations and command-line tools
- **Streamable HTTP**: this replaces the HTTP+SSE transport from previous protocol version. In the Streamable HTTP transport, the server operates as an independent process that can handle multiple client connections. This transport uses HTTP POST and GET requests. Server can optionally make use of Server-Sent Events (SSE) to stream multiple server messages. This permits basic MCP servers, as well as more feature-rich servers supporting streaming and server-to-client notifications and requests.
- **SSE**: legacy transport for HTTP-based MCP servers that use Server-Sent Events transport  

Overall, you should use stdio for building command-line tools, implementing local integrations and working with shell scripts. You should use Streamable HTTP transports when you need a flexible and efficient way for AI agents to communicate with tools and services, especially when dealing with stateless communication or when minimizing resource usage is crucial.

You can also use **custom transports** implementation for your specific needs. 


Let's now connect to the MCP server using stdio transport. First of all, we will use the class `MCPClient` to connect to the [AWS Documentation MCP Server](https://awslabs.github.io/mcp/servers/aws-documentation-mcp-server). This server provides tools to access AWS documentation, search for content, and get recommendations.

In [2]:
# Connect to an MCP server using stdio transport
stdio_mcp_client = MCPClient(
    lambda: stdio_client(
        StdioServerParameters(
            command="uvx", args=["awslabs.aws-documentation-mcp-server@latest"]
        )
    )
)

#### Setup agent configuration and invoke it

Next we will set our agent configuration using the tools from the `stdio_mcp_client` object we just created. To do so, we need to list the tools available in the MCP server. We can use the `list_tools_sync` method for it. 

After that, we will ask a question to our agent.

In [3]:
# Create an agent with MCP tools
with stdio_mcp_client:
    # Get the tools from the MCP server
    tools = stdio_mcp_client.list_tools_sync()

    # Create an agent with these tools
    agent = Agent(
        model="amazon.nova-pro-v1:0",
        tools=tools)

    response = agent("What is Amazon Bedrock pricing model. Be concise.")

<thinking> To find the pricing model for Amazon Bedrock, I should start by searching the AWS documentation for relevant information. I'll use the `search_documentation` tool to find the specific documentation page that details the pricing model for Amazon Bedrock. </thinking>

Tool #1: search_documentation
<thinking> The search results provide several pages related to Amazon Bedrock pricing, but none of them directly state the pricing model. The most relevant result seems to be the one titled "Calculate the cost of running a custom model - Amazon Bedrock," which suggests that the pricing is calculated based on Compute Units (CMUs) and ModelCopy CloudWatch metrics. I should read this section to get a concise understanding of the pricing model. </thinking> 
Tool #2: read_documentation
The Amazon Bedrock pricing model is based on the following components:
1. **Custom Model Units (CMUs):** An abstract view of the hardware use needed to host a single copy of your custom model.
2. **Number o

### Connect to MCP server using Streamable HTTP

Let's now connect to the MCP server using Streamable HTTP transport. First let's start a simple MCP server using Streamable HTTP transport. 

For this example we will create our own MCP server. The architecture will look as following

<div style="text-align:center">
    <img src="images/architecture_2.png" width="65%" />
</div>

In [8]:
# Create an MCP server
mcp = MCPServer("Calculator Server")

# Define a tool


@mcp.tool(description="Calculator tool which performs calculations")
def calculator(x: int, y: int) -> int:
    return x + y


@mcp.tool(description="This is a long running tool")
def long_running_tool(name: str) -> str:
    time.sleep(25)
    return f"Hello {name}"


def main():
    mcp.run(transport="streamable-http")

Let's now start a thread with the `streamable-http` server

In [ ]:
thread = threading.Thread(target=main)
thread.start()

INFO:     Started server process [55653]
INFO:     Waiting for application startup.


[06/22/26 22:58:52] INFO     StreamableHTTP session manager started                  ]8;id=10442267;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=10442268;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/mcp/server/streamable_http_manager.py#131\131]8;;\

INFO:     Application startup complete.
ERROR:    [Errno 48] error while attempting to bind on address ('127.0.0.1', 8000): [errno 48] address already in use
INFO:     Waiting for application shutdown.


                    INFO     StreamableHTTP session manager shutting down            ]8;id=10442274;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=10442275;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/mcp/server/streamable_http_manager.py#135\135]8;;\

INFO:     Application shutdown complete.


#### Integrating Streamable HTTP client with Agent

Now let's use `streamable_http_client` integrate this server with a simple agent. 

In [ ]:
def create_streamable_http_transport():
    return streamable_http_client("http://localhost:8000/mcp")


streamable_http_mcp_client = MCPClient(create_streamable_http_transport)

INFO:     127.0.0.1:65480 - "GET / HTTP/1.1" 404 Not Found


[06/22/26 22:58:33] INFO     Created new transport with session ID:                  ]8;id=10442261;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=10442262;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/mcp/server/streamable_http_manager.py#281\281]8;;\
                             9059bc29a3404ced9ce98fc79706b6c9                                                      

INFO:     127.0.0.1:49394 - "GET /mcp HTTP/1.1" 406 Not Acceptable
INFO:     127.0.0.1:49394 - "GET /favicon.ico HTTP/1.1" 404 Not Found


#### Setup agent configuration and invoke it

Next we will set our agent configuration using the tools from the `streamable_http_mcp_client` object we just created. To do so, we need to list the tools available in the MCP server. We can use the `list_tools_sync` method for it. 

After that, we will ask a question to our agent.

In [13]:
with streamable_http_mcp_client:
    tools = streamable_http_mcp_client.list_tools_sync()

    agent = Agent(
        model="amazon.nova-pro-v1:0",
        tools=tools)

    response = str(agent("What is 2 + 2?"))

[06/22/26 22:59:34] INFO     Created new transport with session ID:                  ]8;id=10442519;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=10442520;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/mcp/server/streamable_http_manager.py#281\281]8;;\
                             cab3f2239bfa4cd9a24b0e031770c892                                                      

INFO:     127.0.0.1:50245 - "POST /mcp HTTP/1.1" 200 OK


                    INFO     HTTP Request: POST http://localhost:8000/mcp "HTTP/1.1 200 OK"         ]8;id=10442525;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=10442526;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     Received session ID: cab3f2239bfa4cd9a24b0e031770c892           ]8;id=10442531;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/mcp/client/streamable_http.py\streamable_http.py]8;;\:]8;id=10442532;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/mcp/client/streamable_http.py#181\181]8;;\

                    INFO     Negotiated protocol version: 2025-11-25                         ]8;id=10442537;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/mcp/client/streamable_http.py\streamable_http.py]8;;\:]8;id=10442538;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/mcp/client/streamable_http.py#193\193]8;;\

...

                    INFO     Processing request of type CallToolRequest                               ]8;id=10442579;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=10442580;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/mcp/server/lowlevel/server.py#733\733]8;;\

The result of 2 + 2 is 4.

[06/22/26 22:59:36] INFO     Terminating session: cab3f2239bfa4cd9a24b0e031770c892           ]8;id=10442585;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/mcp/server/streamable_http.py\streamable_http.py]8;;\:]8;id=10442586;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/mcp/server/streamable_http.py#785\785]8;;\

INFO:     127.0.0.1:50285 - "DELETE /mcp HTTP/1.1" 200 OK


                    INFO     HTTP Request: DELETE http://localhost:8000/mcp "HTTP/1.1 200 OK"       ]8;id=10442591;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=10442592;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/httpx/_client.py#1740\1740]8;;\

### Direct Tool Invocation

While tools are typically invoked by the agent based on user requests, you can also call MCP tools directly. This can be useful for workflow scenarios where you orchestrate multiple tools together.

In [14]:
query = {"x": 10, "y": 20}

with streamable_http_mcp_client:
    # direct tool invocation
    result = streamable_http_mcp_client.call_tool_sync(
        tool_use_id="tool-123", name="calculator", arguments=query
    )

    # Process the result
    print(f"Calculation result: {result['content'][0]['text']}")

[06/22/26 22:59:50] INFO     Created new transport with session ID:                  ]8;id=10442597;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=10442598;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/mcp/server/streamable_http_manager.py#281\281]8;;\
                             61541133e5624c078dda5bd44038ecb1                                                      

INFO:     127.0.0.1:50474 - "POST /mcp HTTP/1.1" 200 OK


                    INFO     HTTP Request: POST http://localhost:8000/mcp "HTTP/1.1 200 OK"         ]8;id=10442603;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=10442604;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     Received session ID: 61541133e5624c078dda5bd44038ecb1           ]8;id=10442609;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/mcp/client/streamable_http.py\streamable_http.py]8;;\:]8;id=10442610;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/mcp/client/streamable_http.py#181\181]8;;\

                    INFO     Negotiated protocol version: 2025-11-25                         ]8;id=10442615;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/mcp/client/streamable_http.py\streamable_http.py]8;;\:]8;id=10442616;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/mcp/client/streamable_http.py#193\193]8;;\

...

                    INFO     Processing request of type ListToolsRequest                              ]8;id=10442651;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=10442652;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/mcp/server/lowlevel/server.py#733\733]8;;\

Calculation result: 30


                    INFO     Terminating session: 61541133e5624c078dda5bd44038ecb1           ]8;id=10442657;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/mcp/server/streamable_http.py\streamable_http.py]8;;\:]8;id=10442658;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/mcp/server/streamable_http.py#785\785]8;;\

INFO:     127.0.0.1:50486 - "DELETE /mcp HTTP/1.1" 200 OK


                    INFO     HTTP Request: DELETE http://localhost:8000/mcp "HTTP/1.1 200 OK"       ]8;id=10442663;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=10442664;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/httpx/_client.py#1740\1740]8;;\

You can optionally also provide `read_timeout_seconds` while calling an MCP server tool to avoid it running for too long

In [15]:
with streamable_http_mcp_client:
    try:
        result = streamable_http_mcp_client.call_tool_sync(
            tool_use_id="tool-123",
            name="long_running_tool",
            arguments={"name": "Amazon"},
            read_timeout_seconds=timedelta(seconds=30),
        )

        if result["status"] == "error":
            print(f"Tool execution failed: {result['content'][0]['text']}")
        else:
            print(f"Tool execution succeeded: {result['content'][0]['text']}")
    except Exception as e:
        print(f"Tool call timed out or failed: {str(e)}")

[06/22/26 23:00:03] INFO     Created new transport with session ID:                  ]8;id=10442669;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=10442670;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/mcp/server/streamable_http_manager.py#281\281]8;;\
                             11e8c583e6a54a48b583e12b2c5906ee                                                      

INFO:     127.0.0.1:50651 - "POST /mcp HTTP/1.1" 200 OK


                    INFO     HTTP Request: POST http://localhost:8000/mcp "HTTP/1.1 200 OK"         ]8;id=10442675;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=10442676;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     Received session ID: 11e8c583e6a54a48b583e12b2c5906ee           ]8;id=10442681;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/mcp/client/streamable_http.py\streamable_http.py]8;;\:]8;id=10442682;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/mcp/client/streamable_http.py#181\181]8;;\

                    INFO     Negotiated protocol version: 2025-11-25                         ]8;id=10442687;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/mcp/client/streamable_http.py\streamable_http.py]8;;\:]8;id=10442688;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/mcp/client/streamable_http.py#193\193]8;;\

...

                    INFO     Processing request of type ListToolsRequest                              ]8;id=10442723;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=10442724;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/mcp/server/lowlevel/server.py#733\733]8;;\

Tool execution succeeded: Hello Amazon


                    INFO     Terminating session: 11e8c583e6a54a48b583e12b2c5906ee           ]8;id=10442729;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/mcp/server/streamable_http.py\streamable_http.py]8;;\:]8;id=10442730;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/mcp/server/streamable_http.py#785\785]8;;\

INFO:     127.0.0.1:50980 - "DELETE /mcp HTTP/1.1" 200 OK


                    INFO     HTTP Request: DELETE http://localhost:8000/mcp "HTTP/1.1 200 OK"       ]8;id=10442735;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=10442736;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/httpx/_client.py#1740\1740]8;;\

### Interacting with multiple MCP servers

With Strands Agents you can also interact with multiple MCP servers using the same agent. Tool execution strategy is configured with the `tool_executor` parameter: agents use `ConcurrentToolExecutor` by default, which runs the model's tool calls in parallel, and `SequentialToolExecutor` runs them one at a time. Let's create a new agent that uses the tools from both servers:

<div style="text-align:center">
    <img src="images/architecture_3.png" width="85%" />
</div>

In this agent, we will again use the AWS Documentation MCP server and we will also use the [AWS IaC MCP Server](https://awslabs.github.io/mcp/servers/aws-iac-mcp-server) which helps with AWS Cloud Development Kit (CDK) best practices, CloudFormation validation, infrastructure as code patterns and security compliance.

First let's connect to the two MCP servers using the stdio transport

In [16]:
# Connect to an MCP server using stdio transport
aws_docs_mcp_client = MCPClient(
    lambda: stdio_client(
        StdioServerParameters(
            command="uvx", args=["awslabs.aws-documentation-mcp-server@latest"]
        )
    )
)

# Connect to the AWS IaC MCP server using stdio transport
iac_mcp_client = MCPClient(
    lambda: stdio_client(
        StdioServerParameters(command="uvx", args=["awslabs.aws-iac-mcp-server@latest"])
    )
)

#### Create Agent with MCP servers

Next we will create the agent with the tools from both MCP servers

In [17]:
# Create an agent with MCP tools
with aws_docs_mcp_client, iac_mcp_client:
    # Get the tools from the MCP server
    tools = aws_docs_mcp_client.list_tools_sync() + iac_mcp_client.list_tools_sync()

    # Create an agent with these tools
    agent = Agent(
        model="amazon.nova-pro-v1:0",
        tools=tools)

    response = agent(
        "What is Amazon Bedrock pricing model. Be concise. Also what are the best practices related to CDK?"
    )

[06/22/26 23:02:47] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=10442741;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=10442742;file:///Users/pathipat/Work/Learning/Yip-aws-strands/samples-strands-agents/.venv/lib/python3.14/site-packages/botocore/credentials.py#1392\1392]8;;\

<thinking> To answer the user's question, I need to gather information on two distinct topics: Amazon Bedrock pricing and CDK best practices. For Amazon Bedrock pricing, I will use the `search_documentation` tool to find relevant AWS documentation. For CDK best practices, I will use the `cdk_best_practices` tool to retrieve the comprehensive guidelines. </thinking>

Tool #1: search_documentation

Tool #2: cdk_best_practices
Amazon Bedrock pricing is based on usage, with different costs for various services and operations. For detailed pricing information, refer to the [Amazon Bedrock pricing page](https://docs.aws.amazon.com/bedrock/latest/userguide/rerank-pricing.html).

Here are some key CDK best practices:

1. **Prefer L2 constructs**: Use L2 constructs for better developer experience and sensible defaults.
2. **Use grant methods**: Leverage `grant*()` methods for permissions instead of manual IAM policies.
3. **Enable encryption by default**: Enable encryption for resources like S3

### MCP is more than tools

This notebook connected to MCP servers to use their **tools**, but tools are only one of the three capabilities an MCP server can expose. Servers can also offer **prompts**, which are reusable, parameterized message templates, and **resources**, which are read-only data addressed by a URI, such as a file or a configuration value. The same `MCPClient` can discover and read them through `list_prompts_sync` and `get_prompt_sync` for prompts, and `list_resources_sync` and `read_resource_sync` for resources. To learn when to reach for each, see the MCP documentation on [prompts](https://modelcontextprotocol.io/docs/concepts/prompts) and [resources](https://modelcontextprotocol.io/docs/concepts/resources). The Strands [`MCPClient` API reference](https://strandsagents.com/docs/api/python/strands.tools.mcp.mcp_client/) documents each of these methods.

## Summary

In this notebook you learned how to connect with MCP servers using Strands Agents and two MCP transport protocols: stdio and Streamable HTTP. You also learned how to connect multiple MCP servers to the same agent. Next, let's see how to use different models with your agent.